# 02. Data Validation & Leakage Verification
Runs `CorpusValidator` (content-quality checks) and `DataLeakageChecker` (zero-overlap audit) against the master corpus and its splits.

In [ ]:
# ============================================================
# PATH BOOSTER — Works on Kineses / Jupyter / Colab / Kaggle
# Handles FileNotFoundError when kernel CWD no longer exists.
# ============================================================
import os, sys

# Step 1: Safely get current directory
try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

# Step 2: Navigate to project root (Kineses home-based path)
kineses_proj = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(kineses_proj):
    os.chdir(kineses_proj)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Step 3: Add project root to Python path
proj_root = os.getcwd()
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')


In [1]:
import os

if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
    if not os.path.exists("Ekegusii-LLM-Translation"):
        os.system("git clone https://github.com/aykahsay/Ekegusii-LLM-Translation.git")
    os.chdir("Ekegusii-LLM-Translation")
    os.system("pip install -q -r requirements.txt")
elif not os.path.exists("src") and os.path.basename(os.getcwd()) == "notebooks":
    # Running locally via `jupyter nbconvert` from within notebooks/ --
    # the repo root (containing src/, data/) is one directory up.
    os.chdir("..")

import sys
sys.path.insert(0, os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


In [2]:
from src.master_corpus.manager import MasterCorpusManager
from src.master_corpus.validator import CorpusValidator
from src.master_corpus.integrity import DataLeakageChecker
from src.utils.constants import SUPPORTED_LANGUAGES

manager = MasterCorpusManager()
validator = CorpusValidator()

## Content validation

In [3]:
sentence_df = manager.load_sentence_corpus()
report = validator.validate(
    sentence_df, 'master_sentence_corpus', 'concept_id', list(SUPPORTED_LANGUAGES),
    max_null_rate=0.30,
)
print(f'Valid: {report.is_valid}')
print(f'Issues: {report.issues}')

INFO | Loaded Master Sentence Corpus: 49,277 concepts.


INFO | [master_sentence_corpus] Validation passed (49,277 rows).


Valid: True
Issues: []


In [4]:
lexical_df = manager.load_lexical_corpus()
lex_report = validator.validate(
    lexical_df, 'master_lexical_corpus', 'lexicon_id', list(SUPPORTED_LANGUAGES),
    max_null_rate=1.0,  # English is 100% empty in this corpus -- see docs/datasets.md
)
print(f'Issues: {lex_report.issues}')

INFO | Loaded Master Lexical Corpus: 268 terms.


INFO | [master_lexical_corpus] Validation passed (268 rows).


Issues: []


## Zero-leakage audit
Every experiment depends on this passing.

In [5]:
checker = DataLeakageChecker(manager)
passed = checker.verify_all()
print(f'Leakage audit passed: {passed}')

INFO | === Starting Master Corpus Data Leakage Audit ===


INFO | Loaded dataset split [master_train.csv]: 39,421 rows.


INFO | Loaded dataset split [master_val.csv]: 4,928 rows.


INFO | Loaded dataset split [master_test.csv]: 4,928 rows.


INFO | 1/2 [ID Audit] Zero Concept ID overlap confirmed across splits.


WARNING | Notice: 746 identical English sentences between Train & Test.


INFO | 2/2 [Text Audit] Sentence text overlap within acceptable limits.


INFO | ✅ [PASSED] 0% Data Leakage Audit Verified Successfully!


Leakage audit passed: True
